# RF + GA — Penjadwalan Pendaftar Bulanan

Alur operasional: pendaftar masuk tanggal 2–9. Jalankan notebook kapan saja setelah data batch tersedia pada bulan pendaftaran; bulan target otomatis menjadi bulan setelah bulan tanggal generate. RF dimuat dari paket model historis; notebook tidak melatih ulang model setiap bulan.

GA memilih tanggal dalam rentang akademik resmi, satu dari empat slot dua jam, dan pasangan penguji. Nama dosen pada tabel harus sama dengan nama kanonis model; tanggal berhalangan kosong berarti tidak tersedia sepanjang waktu. Dosen berhalangan, seminar yang sudah terkunci di bulan target, serta bentrok mahasiswa/pembimbing/penguji diperlakukan sebagai batas wajib.


## 1. Buat templat input
Jalankan sel ini untuk mengunduh empat CSV. Isi data pendaftar bulan ini, termasuk Tanggal_Pendaftaran (tanggal 2–9), rentang kalender akademik, dan periode dosen berhalangan. Nama dosen harus sama dengan nama kanonis pada model RF. Tabel jadwal terkunci boleh kosong tetapi tetap harus memiliki header.


In [ ]:
import pandas as pd
from google.colab import files

templates={
 "template_pendaftar_bulanan.csv":pd.DataFrame(columns=[
  "NIM","Nama","Judul","Jenis_Seminar","Gelombang","topik_model_v4",
  "Pembimbing 1_kanonis","Pembimbing 2_kanonis","Pembimbing 3_kanonis",
  "Tanggal_Pendaftaran"
 ]),
 "template_rentang_kalender.csv":pd.DataFrame([
  ["Sempro","Semua Gelombang",1,12,15,20],
  ["Sidang TA","Semua Gelombang",1,6,22,26],
  ["Sidang TA","Semua Gelombang",10,12,22,26],
 ],columns=["Jenis_Seminar","Gelombang","Bulan_Mulai","Bulan_Akhir","Hari_Mulai","Hari_Akhir"]),
 "template_dosen_berhalangan.csv":pd.DataFrame([
  ["Syamsul Mujahidin","",""],
  ["Tegar Palyus Fiqar","",""],
 ],columns=["Nama_Dosen","Tanggal_Mulai","Tanggal_Akhir"]),
 "template_jadwal_terkunci.csv":pd.DataFrame(columns=[
  "NIM","Nama","Judul","Jenis_Seminar","Tanggal","Waktu",
  "Pembimbing 1_kanonis","Pembimbing 2_kanonis","Pembimbing 3_kanonis",
  "Penguji_1","Penguji_2"
 ]),
}
for name,frame in templates.items():
 path="/content/"+name
 frame.to_csv(path,index=False)
 files.download(path)
print("Kalender dan dosen berhalangan sudah terisi sesuai aturan yang diberikan.")
print("Lengkapi pendaftar batch bulan ini; isi jadwal terkunci bila sudah ada jadwal lain pada bulan target.")


## 2. Atur tanggal generate
Tanggal generate terisi otomatis dengan tanggal hari ini. Kapan pun notebook dijalankan selama bulan batch pendaftaran, target otomatis menjadi bulan berikutnya. Untuk uji data historis, isi `TANGGAL_GENERATE` secara manual.


In [ ]:
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo

# Tidak dibatasi tanggal tertentu: target selalu bulan setelah tanggal generate.
TANGGAL_GENERATE=pd.Timestamp(datetime.now(ZoneInfo("Asia/Makassar")).date())
# Untuk simulasi historis, ganti tanggal di atas, misalnya:
# TANGGAL_GENERATE=pd.Timestamp("2026-08-15")
BULAN_TARGET=(TANGGAL_GENERATE.to_period("M")+1).strftime("%Y-%m")
print("Tanggal generate:",TANGGAL_GENERATE.date())
print("Bulan pelaksanaan target:",BULAN_TARGET)


## 3. Unggah model dan data batch
Unggah satu ZIP paket model historis dan empat CSV templat yang sudah diisi. Berkas jadwal terkunci dapat berisi header saja jika belum ada jadwal lain pada bulan target.


In [ ]:
import io, zipfile, joblib, pandas as pd, numpy as np
from google.colab import files

uploaded=files.upload()
zip_names=[n for n in uploaded if n.lower().endswith(".zip")]
if len(zip_names)!=1: raise ValueError("Unggah tepat satu paket_backend_rf_ga.zip.")
csv_frames={}
for name,payload in uploaded.items():
 if not name.lower().endswith(".csv"): continue
 frame=pd.read_csv(io.BytesIO(payload),dtype=str)
 cols=set(frame.columns)
 if {"NIM","Nama","Judul","Jenis_Seminar","Gelombang","topik_model_v4","Tanggal_Pendaftaran"}.issubset(cols):
  key="batch"
 elif {"Jenis_Seminar","Gelombang","Bulan_Mulai","Bulan_Akhir","Hari_Mulai","Hari_Akhir"}.issubset(cols):
  key="kalender"
 elif {"Nama_Dosen","Tanggal_Mulai","Tanggal_Akhir"}.issubset(cols):
  key="berhalangan"
 elif {"NIM","Tanggal","Waktu","Penguji_1","Penguji_2","Pembimbing 1_kanonis","Pembimbing 2_kanonis","Pembimbing 3_kanonis"}.issubset(cols):
  key="terkunci"
 else: continue
 if key in csv_frames: raise ValueError("Ada CSV ganda untuk "+key)
 csv_frames[key]=frame
required={"batch","kalender","berhalangan","terkunci"}
if set(csv_frames)!=required:
 raise ValueError("Unggah empat CSV: pendaftar, kalender, dosen berhalangan, jadwal terkunci.")
batch,kalender,berhalangan,terkunci=(csv_frames[k].copy() for k in ["batch","kalender","berhalangan","terkunci"])
if batch.empty: raise ValueError("CSV pendaftar bulan ini kosong.")
if batch["NIM"].isna().any() or batch["NIM"].astype(str).str.strip().eq("").any():
 raise ValueError("NIM pendaftar tidak boleh kosong.")
batch["NIM"]=batch["NIM"].astype(str).str.replace(r"\.0$","",regex=True).str.strip()
batch=batch.reset_index(drop=True)
batch.insert(0,"Indeks_Batch",range(len(batch)))
batch["Tanggal_Pendaftaran"]=pd.to_datetime(batch["Tanggal_Pendaftaran"],errors="coerce").dt.normalize()
if batch["Tanggal_Pendaftaran"].isna().any(): raise ValueError("Tanggal_Pendaftaran wajib diisi dengan YYYY-MM-DD.")
periode_daftar=pd.Period(BULAN_TARGET,freq="M")-1
if any(d.to_period("M")!=periode_daftar or d.day<2 or d.day>9 for d in batch["Tanggal_Pendaftaran"]):
 raise ValueError("Batch harus berisi pendaftar tanggal 2–9 pada bulan sebelum BULAN_TARGET.")

# Muat model yang sudah dilatih; tidak ada pelatihan ulang bulanan.
with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as archive:
 model1=joblib.load(io.BytesIO(archive.read("model/penguji_1_target_final.joblib")))
 model2=joblib.load(io.BytesIO(archive.read("model/penguji_2_target_final.joblib")))

FEATURES=["Judul","Jenis_Seminar","topik_model_v4"]
for col in FEATURES:
 batch[col]=batch[col].fillna("Tidak diketahui").astype(str).str.strip()
 batch.loc[batch[col].eq(""),col]="Tidak diketahui"
X=batch[FEATURES].copy()

def add_top5(frame,model,prefix):
 proba=model.predict_proba(X)
 classes=model.named_steps["rf"].classes_
 top=min(5,len(classes))
 order=np.argsort(proba,axis=1)[:,::-1][:,:top]
 for rank in range(top):
  frame[f"{prefix}_kandidat_{rank+1}"]=[classes[j] for j in order[:,rank]]
  frame[f"{prefix}_skor_{rank+1}"]=[float(proba[i,j]) for i,j in enumerate(order[:,rank])]
add_top5(batch,model1,"P1")
add_top5(batch,model2,"P2")
known=set()
for c in ["Pembimbing 1_kanonis","Pembimbing 2_kanonis","Pembimbing 3_kanonis"]:
 for v in batch[c].dropna():
  if str(v).strip(): known.add(" ".join(str(v).strip().casefold().split()))
for model in [model1,model2]:
 for v in model.named_steps["rf"].classes_:
  if str(v).strip(): known.add(" ".join(str(v).strip().casefold().split()))
unknown_absence=sorted({" ".join(str(v).strip().casefold().split()) for v in csv_frames["berhalangan"]["Nama_Dosen"].dropna() if str(v).strip()}-known)
if unknown_absence:
 raise ValueError("Nama dosen berhalangan tidak cocok dengan nama kanonis model/batch: "+", ".join(unknown_absence))
batch.to_csv("/content/kandidat_batch_bulanan.csv",index=False)
print("Pendaftar batch:",len(batch))
print("Kandidat RF dibuat untuk batch ini saja.")


## 4. GA dan audit jadwal bulan target
Rentang kalender difilter ke BULAN_TARGET. Slot tetap: 08.00–10.00, 10.00–12.00, 13.00–15.00, dan 15.00–17.00. Jika tidak ditemukan solusi tanpa bentrok, notebook menghentikan ekspor.


In [ ]:
import random
from collections import defaultdict
from itertools import combinations
from IPython.display import display

def norm(v):
 if pd.isna(v): return ""
 return " ".join(str(v).strip().casefold().split())
def nimkey(v):
 if pd.isna(v): return ""
 s=str(v).strip()
 if s.endswith(".0"): s=s[:-2]
 return "".join(ch for ch in s if ch.isdigit())
def parse_date_col(frame,col):
 frame[col]=pd.to_datetime(frame[col],errors="coerce").dt.normalize()
 if frame[col].isna().any(): raise ValueError("Tanggal tidak valid pada "+col+"; gunakan YYYY-MM-DD.")
# Kalender adalah aturan berulang: bulan dan hari, tidak terikat tahun.
for c in ["Bulan_Mulai","Bulan_Akhir","Hari_Mulai","Hari_Akhir"]:
 kalender[c]=pd.to_numeric(kalender[c],errors="raise").astype(int)
if ((kalender["Bulan_Mulai"]<1)|(kalender["Bulan_Akhir"]>12)|(kalender["Bulan_Akhir"]<kalender["Bulan_Mulai"])).any():
 raise ValueError("Bulan kalender harus 1–12 dan urut.")
if ((kalender["Hari_Mulai"]<1)|(kalender["Hari_Akhir"]>31)|(kalender["Hari_Akhir"]<kalender["Hari_Mulai"])).any():
 raise ValueError("Hari kalender tidak valid.")
kalender["jenis_key"]=kalender["Jenis_Seminar"].map(norm)
kalender["wave_key"]=kalender["Gelombang"].map(norm)
target_period=pd.Period(BULAN_TARGET,freq="M")
target_start=target_period.start_time.normalize()
target_end=target_period.end_time.normalize()
month_num=target_period.month
kalender_target=kalender[(kalender["Bulan_Mulai"]<=month_num)&(kalender["Bulan_Akhir"]>=month_num)].copy()
if kalender_target.empty: raise ValueError("Tidak ada aturan kalender berulang untuk bulan target "+BULAN_TARGET)
# Date fields may both be blank to mark a lecturer unavailable indefinitely.
start_blank=berhalangan["Tanggal_Mulai"].fillna("").astype(str).str.strip().eq("")
end_blank=berhalangan["Tanggal_Akhir"].fillna("").astype(str).str.strip().eq("")
if (start_blank != end_blank).any(): raise ValueError("Isi kedua tanggal dosen berhalangan atau kosongkan keduanya untuk larangan permanen.")
for c in ["Tanggal_Mulai","Tanggal_Akhir"]:
 raw=berhalangan[c].fillna("").astype(str).str.strip()
 parsed=pd.to_datetime(raw.where(raw.ne("")),errors="coerce").dt.normalize()
 if (raw.ne("")&parsed.isna()).any(): raise ValueError("Tanggal dosen berhalangan tidak valid; gunakan YYYY-MM-DD.")
 berhalangan[c]=parsed
berhalangan["dosen_key"]=berhalangan["Nama_Dosen"].map(norm)

SLOTS=[("Sesi 1","08:00","10:00"),("Sesi 2","10:00","12:00"),("Sesi 3","13:00","15:00"),("Sesi 4","15:00","17:00")]
def minute(t):
 h,m=map(int,t.split(":")); return 60*h+m
def parse_interval(v):
 parts=str(v).strip().replace(".",":").replace("–","-").split("-")
 if len(parts)!=2: raise ValueError("Format waktu harus HH:MM - HH:MM: "+str(v))
 a,b=[x.strip() for x in parts]
 return minute(a),minute(b)
def absent(name,date):
 k=norm(name)
 for _,r in berhalangan[berhalangan["dosen_key"]==k].iterrows():
  start,end=r["Tanggal_Mulai"],r["Tanggal_Akhir"]
  if pd.isna(start) and pd.isna(end): return True
  if start<=date<=end: return True
 return False
SUPCOLS=["Pembimbing 1_kanonis","Pembimbing 2_kanonis","Pembimbing 3_kanonis"]
for c in SUPCOLS:
 if c not in batch.columns: raise ValueError("Kolom pembimbing kanonis wajib ada: "+c)
def sups(row): return {norm(row[c]) for c in SUPCOLS if norm(row[c])}
def candidates(row,prefix):
 out=[]; seen=set()
 for n in range(1,6):
  name=row.get(f"{prefix}_kandidat_{n}"); key=norm(name)
  if not key or key in seen: continue
  seen.add(key)
  try: score=float(row.get(f"{prefix}_skor_{n}",0))
  except (TypeError,ValueError): score=0.0
  out.append((str(name).strip(),key,score))
 return out

# Validate locked schedules and build their fixed participant/time records.
fixed=[]
for _,r in terkunci.iterrows():
 if pd.isna(r["Tanggal"]) or pd.isna(r["Waktu"]): raise ValueError("Jadwal terkunci harus memiliki Tanggal dan Waktu.")
 if not nimkey(r["NIM"]): raise ValueError("NIM jadwal terkunci wajib diisi.")
 if not norm(r["Penguji_1"]) or not norm(r["Penguji_2"]): raise ValueError("Kedua penguji jadwal terkunci wajib diisi.")
 date=pd.to_datetime(r["Tanggal"],errors="coerce")
 if pd.isna(date): raise ValueError("Tanggal jadwal terkunci tidak valid.")
 start,end=parse_interval(r["Waktu"])
 if end<=start: raise ValueError("Urutan waktu jadwal terkunci tidak valid.")
 people={"M:"+nimkey(r["NIM"])}
 for c in SUPCOLS:
  if c in terkunci.columns and norm(r.get(c)): people.add("D:"+norm(r[c]))
 for c in ["Penguji_1","Penguji_2"]:
  if norm(r.get(c)): people.add("D:"+norm(r[c]))
 fixed.append({"date":date.normalize(),"start":start,"end":end,"people":people})
for f in fixed:
 unavailable_fixed=[person[2:] for person in f["people"] if person.startswith("D:") and absent(person[2:],f["date"])]
 if unavailable_fixed: raise ValueError("Jadwal terkunci mencakup dosen berhalangan: "+", ".join(unavailable_fixed))
for i,a in enumerate(fixed):
 for b in fixed[i+1:]:
  if a["date"]==b["date"] and a["start"]<b["end"] and b["start"]<a["end"] and a["people"]&b["people"]:
   raise ValueError("Jadwal terkunci yang diunggah sudah memiliki bentrok peserta; perbaiki sebelum menjadwalkan batch baru.")

def overlaps(a,b): return a["start"]<b["end"] and b["start"]<a["end"]
def conflicts_fixed(date,start,end,people):
 slot={"date":date,"start":start,"end":end}
 return any(f["date"]==date and overlaps(slot,f) and people&f["people"] for f in fixed)

# Map each registration to its approved date range and feasible RF pairs.
options={}
no_options=[]
for ri,row in batch.iterrows():
 kind,wave=norm(row["Jenis_Seminar"]),norm(row["Gelombang"])
 matching=kalender_target[(kalender_target["jenis_key"]==kind)&(kalender_target["wave_key"].isin([wave,"semua gelombang","semua","*"]))]
 exact=matching[matching["wave_key"]==wave]
 windows=exact if not exact.empty else matching
 if windows.empty:
  no_options.append((ri,"gelombang tidak punya rentang pada bulan target")); continue
 supervisors=sups(row)
 pairs=[]
 for a,ak,sa in candidates(row,"P1"):
  for b,bk,sb in candidates(row,"P2"):
   if ak!=bk and ak not in supervisors and bk not in supervisors:
    pairs.append((a,ak,b,bk,sa+sb))
 opts={}
 for _,w in windows.iterrows():
  first_day=int(w["Hari_Mulai"]); last_day=int(w["Hari_Akhir"])
  try:
   first=pd.Timestamp(target_period.year,target_period.month,first_day)
   last=pd.Timestamp(target_period.year,target_period.month,last_day)
  except ValueError as e:
   raise ValueError("Hari pada rentang kalender tidak ada di bulan target: "+str(e))
  for date in pd.date_range(first,last,freq="D"):
   date=date.normalize()
   if any(absent(s,date) for s in supervisors): continue
   for si,(label,t1,t2) in enumerate(SLOTS):
    start,end=minute(t1),minute(t2)
    for a,ak,b,bk,score in pairs:
     if absent(a,date) or absent(b,date): continue
     student=nimkey(row["NIM"])
     people={"M:"+student}|{"D:"+s for s in supervisors|{ak,bk}}
     if conflicts_fixed(date,start,end,people): continue
     key=(date.strftime("%Y-%m-%d"),si,ak,bk)
     opts[key]={"date":date,"si":si,"slot":label,"time":f"{t1} - {t2}","start":start,"end":end,"p1":a,"p2":b,"score":score,"people":people}
 options[ri]=sorted(opts.values(),key=lambda x:x["score"],reverse=True)
 if not options[ri]: no_options.append((ri,"tidak ada slot/pasangan yang memenuhi aturan"))
if no_options:
 display(pd.DataFrame([{"Baris_Batch":i,"Alasan":why} for i,why in no_options]))
 raise RuntimeError("Ada pendaftar yang belum bisa dijadwalkan dalam bulan target.")

# GA: minimalkan bentrok keras; di antara solusi valid, maksimalkan skor RF.
random.seed(42); rows=list(batch.index)
def keyopt(o): return (o["date"].strftime("%Y-%m-%d"),o["si"])
def fitness(ind):
 used=defaultdict(set); clash=0; score=0.0
 for ri,j in zip(rows,ind):
  o=options[ri][j]; k=keyopt(o)
  for person in o["people"]:
   if person in used[k]: clash+=1
   used[k].add(person)
  score+=o["score"]
 return clash,score
def seed_ind():
 order=rows.copy(); random.shuffle(order); order.sort(key=lambda r:len(options[r]))
 used=defaultdict(set); chosen={}
 for ri in order:
  ranked=[]
  for j,o in enumerate(options[ri]):
   n=sum(p in used[keyopt(o)] for p in o["people"])
   ranked.append((n,-o["score"],j))
  low=min(x[0] for x in ranked); pool=sorted(x for x in ranked if x[0]==low)[:20]
  j=random.choice(pool)[2]; chosen[ri]=j; used[keyopt(options[ri][j])].update(options[ri][j]["people"])
 return [chosen[r] for r in rows]
POP,GEN,ELITE,MUT=30,50,5,0.02
population=[seed_ind() for _ in range(POP)]
best=None; bestfit=(float("inf"),-float("inf"))
for _ in range(GEN):
 ranked=sorted(((fitness(x),x) for x in population),key=lambda z:(z[0][0],-z[0][1]))
 if (ranked[0][0][0],-ranked[0][0][1])<bestfit:
  bestfit=(ranked[0][0][0],-ranked[0][0][1]); best=ranked[0][1][:]
 elites=[x[:] for _,x in ranked[:ELITE]]; parents=[x for _,x in ranked[:max(ELITE*3,10)]]
 nxt=elites[:]
 while len(nxt)<POP:
  a,b=random.choice(parents),random.choice(parents)
  cut=random.randrange(1,len(rows)) if len(rows)>1 else 0
  child=a[:cut]+b[cut:]
  for g,ri in enumerate(rows):
   if random.random()<MUT: child[g]=random.randrange(len(options[ri]))
  nxt.append(child)
 population=nxt
ranked=sorted(((fitness(x),x) for x in population),key=lambda z:(z[0][0],-z[0][1]))
if (ranked[0][0][0],-ranked[0][0][1])<bestfit:
 bestfit=(ranked[0][0][0],-ranked[0][0][1]); best=ranked[0][1][:]
print("Batch pendaftar:",len(rows),"bulan target:",BULAN_TARGET,"bentrok terbaik:",bestfit[0])
if bestfit[0]: raise RuntimeError("GA belum menemukan jadwal bebas bentrok; tidak ada ekspor.")

# Build schedule and audit every new assignment against each other and fixed schedules.
records=[]
for pos,ri in enumerate(rows):
 row=batch.loc[ri]; o=options[ri][best[pos]]
 rec=row.to_dict()
 rec.update({"Tanggal":o["date"].strftime("%Y-%m-%d"),"Slot":o["slot"],"Waktu":o["time"],"Penguji_1":o["p1"],"Penguji_2":o["p2"],"Skor_RF_Pasangan":o["score"]})
 rec["_people"]=o["people"]; rec["_start"]=o["start"]; rec["_end"]=o["end"]
 records.append(rec)
audit=[]
# Pemeriksaan ulang aturan per jadwal: bulan target, rentang resmi, slot, dan dosen.
for pos,ri in enumerate(rows):
 row=batch.loc[ri]; opt=options[ri][best[pos]]
 date=opt["date"]
 if date.to_period("M").strftime("%Y-%m")!=BULAN_TARGET:
  audit.append((ri,"tanggal_bukan_bulan_target"))
 kind,wave=norm(row["Jenis_Seminar"]),norm(row["Gelombang"])
 matching=kalender_target[(kalender_target["jenis_key"]==kind)&(kalender_target["wave_key"].isin([wave,"semua gelombang","semua","*"]))]
 exact=matching[matching["wave_key"]==wave]
 win=exact if not exact.empty else matching
 day_rules=win[(win["Hari_Mulai"]<=date.day)&(win["Hari_Akhir"]>=date.day)]
 if day_rules.empty: audit.append((ri,"tanggal_di_luar_rentang_resmi"))
 label,t1,t2=SLOTS[opt["si"]]
 if minute(t2)-minute(t1)!=120 or opt["time"]!=f"{t1} - {t2}": audit.append((ri,"slot_bukan_dua_jam"))
 p1,p2=norm(opt["p1"]),norm(opt["p2"]); supervisors=sups(row)
 if not p1 or not p2 or p1==p2: audit.append((ri,"pasangan_penguji_tidak_valid"))
 if p1 in supervisors or p2 in supervisors: audit.append((ri,"pembimbing_menjadi_penguji"))
 if any(absent(name,date) for name in supervisors|{p1,p2}): audit.append((ri,"dosen_berhalangan"))
for a in records:
 date=pd.Timestamp(a["Tanggal"])
 for f in fixed:
  if date==f["date"] and a["_start"]<f["end"] and f["start"]<a["_end"] and a["_people"]&f["people"]:
   audit.append((a["Indeks_Batch"],"bentrok_dengan_jadwal_terkunci"))
for i,a in enumerate(records):
 for b in records[i+1:]:
  if a["Tanggal"]==b["Tanggal"] and a["_start"]<b["_end"] and b["_start"]<a["_end"]:
   shared=a["_people"]&b["_people"]
   if shared: audit.append((a["Indeks_Batch"],b["Indeks_Batch"],sorted(shared)))
for a in records:
 date=pd.Timestamp(a["Tanggal"])
 if norm(a["Penguji_1"])==norm(a["Penguji_2"]): audit.append((a["Indeks_Batch"],"pasangan sama"))
 if norm(a["Penguji_1"]) in sups(a) or norm(a["Penguji_2"]) in sups(a): audit.append((a["Indeks_Batch"],"pembimbing menjadi penguji"))
 if any(absent(n,date) for n in sups(a)|{norm(a["Penguji_1"]),norm(a["Penguji_2"])}): audit.append((a["Indeks_Batch"],"dosen berhalangan"))
if audit: raise RuntimeError("Audit akhir gagal; jadwal tidak diekspor.")
jadwal_bulanan=pd.DataFrame([{k:v for k,v in r.items() if not k.startswith("_")} for r in records])
jadwal_bulanan.to_csv("/content/jadwal_bulanan_valid.csv",index=False,encoding="utf-8-sig")
jadwal_bulanan.to_excel("/content/jadwal_bulanan_valid.xlsx",index=False)
print("Audit lolos. Bentrok dengan jadwal terkunci dan dalam batch: 0.")
files.download("/content/jadwal_bulanan_valid.csv")
files.download("/content/jadwal_bulanan_valid.xlsx")


## 5. Paket hasil bulanan
Simpan jadwal ini sebagai jadwal terkunci untuk batch berikutnya jika waktunya masih berada pada bulan target. Dengan begitu, gelombang lain yang dijadwalkan pada bulan yang sama tetap diperiksa terhadap peserta yang sudah terjadwal.


In [ ]:
import os, zipfile
from google.colab import files
if "jadwal_bulanan" not in globals(): raise RuntimeError("Jalankan GA dan audit terlebih dahulu.")
rules_dir="/content/aturan_bulanan"; os.makedirs(rules_dir,exist_ok=True)
batch.to_csv(f"{rules_dir}/pendaftar_batch.csv",index=False)
kalender.to_csv(f"{rules_dir}/rentang_kalender.csv",index=False)
berhalangan.to_csv(f"{rules_dir}/dosen_berhalangan.csv",index=False)
terkunci.to_csv(f"{rules_dir}/jadwal_sebelumnya_terkunci.csv",index=False)
zip_path="/content/paket_jadwal_bulanan.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
 z.write("/content/jadwal_bulanan_valid.csv","hasil/jadwal_bulanan_valid.csv")
 z.write("/content/jadwal_bulanan_valid.xlsx","hasil/jadwal_bulanan_valid.xlsx")
 z.write("/content/kandidat_batch_bulanan.csv","hasil/kandidat_batch_bulanan.csv")
 for name in os.listdir(rules_dir): z.write(f"{rules_dir}/{name}",f"aturan/{name}")
print("Paket hasil bulanan:",zip_path)
files.download(zip_path)


## Penggunaan rutin
Isi batch pendaftar bertanggal 2–9 dan jalankan notebook kapan saja setelah batch tersedia pada bulan pendaftaran. Target otomatis menjadi bulan berikutnya. Unggah kalender akademik, daftar dosen berhalangan, jadwal bulan target yang sudah terkunci, serta ZIP model historis. GA hanya menjadwalkan batch baru untuk bulan target. Setelah audit lolos, gunakan hasil tersebut sebagai jadwal terkunci pada proses berikutnya bila rentang waktunya masih relevan.
